In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Analyse exploratoire des données

### Distribution

In [ ]:
path = 'C:/Users/plled/Documents/SN2/PE/VMMRdb'

#On récupère toutes les marques du dataset
marques = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]


#On compte le nombres d'images par marques
nb_pictures = []
for brand in os.listdir(path):
    brand_path=os.path.join(path, brand)
    if os.path.isdir(brand_path):
        num_images = len([f for f in os.listdir(brand_path) if os.path.isfile(os.path.join(brand_path, f))])
        nb_pictures.append(num_images)

#On affiche les résultats
plt.figure()
sns.barplot(x=nb_pictures, y=marques)
plt.title("Nombre d'images par marques")
plt.xlabel("Quantité")
plt.ylabel("Marques")
plt.show()


### Format des images

In [ ]:
from PIL import Image


base_path = 'C:/Users/plled/Documents/SN2/PE/VMMRdb' 
data_list = []


for marque in os.listdir(base_path):
    marque_path = os.path.join(base_path, marque)
    
    if os.path.isdir(marque_path):
        for img_name in os.listdir(marque_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(marque_path, img_name)
                
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        ratio = w / h
                        
                        data_list.append({
                            'marque': marque,
                            'nom': img_name,
                            'largeur': w,
                            'hauteur': h,
                            'ratio': round(ratio, 2)
                        })
                except Exception as e:
                    print(f"Erreur sur l'image {img_name}: {e}")

# 4. Création du DataFrame
df = pd.DataFrame(data_list)

# 5. Affichage des statistiques globales
print("\n--- Statistiques Générales ---")
print(df[['largeur', 'hauteur', 'ratio']].describe())

# 6. Repérer les "outliers" (images avec un ratio bizarre)
# Exemple : images beaucoup plus larges que hautes (ratio > 2)
print("\n--- Images très larges (Limo ou panoramique ?) ---")
print(df[df['ratio'] > 2.0].head())


### Sanity Check

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

# 1. Configuration
base_path = 'C:/Users/plled/Documents/SN2/PE/VMMRdb'
nb_marques_a_voir = 4  # Nombre de lignes
images_par_marque = 4  # Nombre de colonnes

# 2. Récupérer la liste des marques (dossiers)
marques = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]

# Sélectionner quelques marques au hasard pour l'inspection
marques_selectionnees = random.sample(marques, nb_marques_a_voir)

# 3. Création de la figure
fig, axes = plt.subplots(nb_marques_a_voir, images_par_marque, figsize=(15, 10))

for i, marque in enumerate(marques_selectionnees):
    marque_path = os.path.join(base_path, marque)
    
    # Lister les images dans ce dossier
    toutes_images = [f for f in os.listdir(marque_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    # Sélectionner des images au hasard
    echantillon = random.sample(toutes_images, min(images_par_marque, len(toutes_images)))
    
    for j, img_nom in enumerate(echantillon):
        img_path = os.path.join(marque_path, img_nom)
        
        try:
            img = Image.open(img_path)
            ax = axes[i, j]
            ax.imshow(img)
            ax.set_title(f"{marque}")
            ax.axis('off') # On cache les axes pour que ce soit plus propre
        except Exception as e:
            print(f"Erreur sur {img_nom}: {e}")

plt.tight_layout()
plt.show()

# Detecter les voitures avec ResNet-50

In [ ]:
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

# Détection du GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {torch.cuda.get_device_name(device) if torch.cuda.is_available() else 'CPU'}")

# Chargement du modèle ResNet50 avec les poids ImageNet
ResNet50_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
ResNet50_model = ResNet50_model.to(device)
ResNet50_model.eval() # On le met en mode évaluation tout de suite

# Elle redimensionne, convertit en 0-1 (ToTensor) et applique la normalisation ImageNet
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### Preprocessing

ResNet prend en entrée un tenseur 4D (un vecteur de cube de matrices), c'est à dire une grille d'image au format 224*224. Il faut donc les redimensionner. Le problème potentiel est que cette opération va déformer les images, ce qui peut induire le modèle en erreur. On verra pour une amélioration après les résultats des premiers test.





In [ ]:
def preprocess_image(img_path):
    """
    Transforme une image disque en un tenseur 4D (1, 3, 224, 224) prêt pour le GPU.
    """
    # 1. Charge l'image et s'assure du format RGB
    img = Image.open(img_path).convert('RGB')
    
    # 2. Applique la recette (Resize + ToTensor + Normalize)
    x = img_transform(img)
    
    # 3. Ajoute la dimension de batch (1, 3, 224, 224)
    return x.unsqueeze(0)

def paths_to_tensor(img_paths):
    """
    Prend une liste de chemins et les transforme en un seul gros bloc (Batch).
    """
    list_of_tensors = [preprocess_image(img_path) for img_path in tqdm(img_paths)]
    
    # On empile tout le monde sur la première dimension
    return torch.cat(list_of_tensors, dim=0)



ResNet renvoie un vecteur avec les probabilités d'appartenance aux 1000 classes d'ImageNet.

### Fonction de détection des voitures

In [ ]:
def car_detector(img_path):
    """
    Détecte si l'image à img_path contient une voiture.
    """
    # On récupère le tenseur déjà préparé
    img_tensor = preprocess_image(img_path).to(device)
    
    # Inférence
    with torch.no_grad():
        output = ResNet50_model(img_tensor)
        indice_predit = torch.argmax(output).item()
    
    # Vérification des indices car/truck/etc d'ImageNet
    car_indices = [436, 468, 511, 603, 627, 656, 751, 817]
    return indice_predit in car_indices

### Vérification du dataset


In [ ]:
from PIL import Image


base_path = 'C:/Users/plled/Documents/SN2/PE/VMMRdb' 
not_car_counter = 0

for marque in os.listdir(base_path):
    marque_path = os.path.join(base_path, marque)
    
    if os.path.isdir(marque_path):
        for img_name in os.listdir(marque_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(marque_path, img_name)
                if not car_detector(img_path):
                    print("Objet autre détecté : ", img_path)
                    not_car_counter+=1
                                    

# Détecter les voitures avec YOLO

In [ ]:
import os
import cv2
from ultralytics import YOLO
from tqdm import tqdm

# Configuration des chemins
base_path = r'C:/Users/plled/Documents/SN2/PE/VMMRdb'
# IDs des classes COCO pour les véhicules : 2 (car), 5 (bus), 7 (truck)
target_classes = [2, 5, 7]
conf_threshold = 0.5

# Chargement du modèle sur la RTX 4050 (device=0)
# Le modèle nano est choisi pour sa vitesse sur 82 000 fichiers
model = YOLO('yolo11n.pt')

valid_cars = 0
invalid_files = 0
total_processed = 0

# Liste de tous les chemins d'images pour la barre de progression
image_paths = []
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(root, file))

print(f"Début de l'analyse de {len(image_paths)} fichiers sur GPU...")

# Boucle de traitement avec barre de progression
for img_path in tqdm(image_paths, desc="Filtrage YOLO"):
    total_processed += 1
    
    # Inférence sur GPU
    results = model(img_path, device=0, verbose=False, conf=conf_threshold)
    
    found_vehicle = False
    for r in results:
        # On vérifie si l'une des détections appartient à nos classes cibles
        detected_classes = r.boxes.cls.tolist()
        if any(cls in target_classes for cls in detected_classes):
            found_vehicle = True
            break
            
    if found_vehicle:
        valid_cars += 1
    else:
        invalid_files += 1
        # Optionnel : log des fichiers non reconnus
        # print(f"Non-véhicule ou confiance faible : {img_path}")

# Résumé des statistiques
print("\n--- Rapport Final de Détection ---")
print(f"Total images analysées : {total_processed}")
print(f"Véhicules confirmés : {valid_cars}")
print(f"Images écartées : {invalid_files}")
print(f"Taux de validité : {(valid_cars/total_processed)*100:.2f}%")

# Modèle

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models
import os

### train/validation split

In [5]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# On definit le chemin
base_path = 'C:/Users/plled/Documents/SN2/PE/VMMRdb'

# Parametres de chargement
batch_size = 32  # Taille optimale pour 6Go de VRAM
img_size = (224, 224)
val_split = 0.2
seed = 123

# Transformations appliquees aux images
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor()
])

# Chargement du dataset complet
full_dataset = datasets.ImageFolder(root=base_path, transform=transform)

# Split train/validation (80/20)
num_samples = len(full_dataset)
num_val = int(val_split * num_samples)
num_train = num_samples - num_val

generator = torch.Generator().manual_seed(seed)
train_ds, val_ds = random_split(full_dataset, [num_train, num_val], generator=generator)

# DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=batch_size, #taille du batch 
    shuffle=True, #mélange les données à chaque epoch pour éviter l'overfitting
    num_workers=0, #préparateurs d'images
    pin_memory=torch.cuda.is_available() 
 )
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
 )

# Recuperation automatique du nombre de marques
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Nombre de marques detectees : {num_classes}")

Nombre de marques detectees : 23


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import copy

# 1. On prépare le modèle
def build_car_model(num_classes):
    # Charger ResNet50 avec les poids ImageNet
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    
    # Geler les poids (base_model.trainable = False)
    for param in model.parameters():
        param.requires_grad = False
    
    # Remplacer la dernière couche (la "tête")
    # En PyTorch, le pooling est déjà géré juste avant 'fc'
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(num_ftrs, num_classes) # Pas de Softmax ici ! CrossEntropyLoss le fait tout seul
    )
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_car_model(num_classes).to(device)

# 2. Compilation (Optimizer et Loss)
criterion = nn.CrossEntropyLoss() # Équivalent categorical_crossentropy
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) # On n'entraîne QUE la nouvelle tête

In [ ]:
from tqdm import tqdm
import sys 


epochs = 10
patience = 3 #nombre d'époques maximales sans amélioration
best_val_loss = float('inf') #on initialise la meilleure perte à l'infini pour la première comparaison
patience_counter = 0 #compteur d'époques sans amélioration
best_model_wts = copy.deepcopy(model.state_dict()) #photocopie des poids du modèle pour sauvegarder le meilleur

print("Démarrage de l'entraînement...")

for epoch in range(epochs): #on lance la boucle d'entraînement pour le nombre d'époques défini

    # --- PHASE D'ENTRAÎNEMENT ---

    model.train() #on met le modèle en mode entraînement pour activer les Dropout et BatchNorm
    train_loss = 0.0 #compteur de perte pour l'époque en cours

    train_bar = tqdm(train_loader, desc=f"Époque {epoch+1}/{epochs} [Train]", file=sys.stdout)
    
    for inputs, labels in train_loader: #on itère sur les batches du DataLoader. Chaque "inputs" est un batch d'images et "labels" les classes correspondantes
        inputs, labels = inputs.to(device), labels.to(device) #on envoie les données sur le GPU
        
        optimizer.zero_grad()           # on efface les gradients calculés lors de la passe précédente
        outputs = model(inputs)         # on donne les images au modèle pour obtenir les prédictions
        loss = criterion(outputs, labels) #calcul de la perte
        loss.backward()                 # Backward (Calcul des gradients)
        optimizer.step()                # Mise à jour des poids
        
        train_loss += loss.item() * inputs.size(0) #perte totale de l'époque (on multiplie par la taille du batch pour avoir la perte totale et pas la moyenne)

    # --- PHASE DE VALIDATION ---

    model.eval() #on met le modèle en mode évaluation pour désactiver les Dropout et BatchNorm
    #initialisation des compteurs de perte et de précision pour la validation
    val_loss = 0.0 
    corrects = 0
    
    with torch.no_grad(): #on désactive le calcul des gradients pour accélérer la validation et économiser de la mémoire
        for inputs, labels in val_loader: #on itère sur les batches de validation
            inputs, labels = inputs.to(device), labels.to(device) #on envoie les données sur le GPU
            outputs = model(inputs) #on donne les images au modèle pour obtenir les prédictions
            loss = criterion(outputs, labels) #calcul de la perte
            val_loss += loss.item() * inputs.size(0) #on accumule la perte totale de validation
            
            _, preds = torch.max(outputs, 1) #torch.max retourne la valeur max et son indice. On prend l'indice qui correspond à la classe prédite
            corrects += torch.sum(preds == labels.data) #on compare les prédictions avec les labels réels et on compte le nombre de bonnes réponses

    epoch_loss = val_loss / len(val_loader.dataset) #perte moyenne par image pour la validation
    epoch_acc = corrects.double() / len(val_loader.dataset) #précision moyenne pour la validation
    
    print(f"Epoch {epoch+1}/{epochs} | Val Loss: {epoch_loss:.4f} | Val Acc: {epoch_acc:.4f}") #affichage de la perte et de la précision à chaque époque

    # --- LOGIQUE DES CALLBACKS (Checkpoint & Early Stopping) ---
    if epoch_loss < best_val_loss: #si la perte de validation s'améliore, on sauvegarde le modèle et on réinitialise le compteur de patience
        best_val_loss = epoch_loss #on met à jour la meilleure perte
        best_model_wts = copy.deepcopy(model.state_dict()) #on sauvegarde les poids du modèle actuel comme le meilleur
        torch.save(model.state_dict(), 'best_car_model.pth') # on sauvegarde les poids du meilleur modèle sur le disque (ModelCheckpoint)
        patience_counter = 0 #on réinitialise le compteur de patience car on a une amélioration
        print("Nouveau meilleur modèle sauvegardé !")
    else:
        patience_counter += 1 #si pas d'amélioration, on incrémente le compteur de patience
        if patience_counter >= patience: #si on a atteint le nombre maximum d'époques sans amélioration, on arrête l'entraînement
            print("Early Stopping : La validation ne s'améliore plus.")
            break

# Charger les meilleurs poids à la fin
model.load_state_dict(best_model_wts) #on remet les poids du meilleur modèle trouvé pendant l'entraînement pour pouvoir l'utiliser ensuite (évaluation finale, export, etc)

c:\Users\plled\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Démarrage de l'entraînement...


KeyboardInterrupt: 